In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import recall_score, f1_score, roc_auc_score
from models import *
from datasets import *

In [2]:
batch_size = 512

dataset = BinaryDataset('Node123_loop')
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(20))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [3]:
epochs = 500
learning_rate = 0.005

# 初始化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CnnClassifier().to(device)
# model = CnnClassifierPairwise().to(device)
# model = MLPClassifier(dataset.n_buses).to(device)
# model = TransformerClassifier(dataset.n_buses).to(device)
# model = LSTMClassifier(dataset.n_buses).to(device)

# 定义损失函数和优化器
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [4]:
model

CnnClassifier(
  (features): Sequential(
    (0): Conv1d(1, 32, kernel_size=(2,), stride=(1,), padding=(1,))
    (1): ReLU(inplace=True)
    (2): Conv1d(32, 64, kernel_size=(2,), stride=(1,), padding=(1,))
    (3): ReLU(inplace=True)
    (4): Conv1d(64, 128, kernel_size=(2,), stride=(1,), padding=(1,))
    (5): ReLU(inplace=True)
  )
  (classifier): Sequential(
    (0): LazyLinear(in_features=0, out_features=64, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [5]:
# 训练模型
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # 在测试集上计算 loss
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
    
    test_loss /= len(test_loader)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {running_loss/len(train_loader):.4f}, Test Loss: {test_loss:.4f}")

    # 如果测试集损失比上一轮小，则保存模型
    if epoch == 0 or running_loss < best_test_loss:
        best_test_loss = running_loss
        torch.save(model.state_dict(), 'voltage_cnn_model.pth')

Epoch 1/500, Train Loss: 1.9263, Test Loss: 0.7012
Epoch 2/500, Train Loss: 0.7030, Test Loss: 0.7034
Epoch 3/500, Train Loss: 0.6951, Test Loss: 0.6999
Epoch 4/500, Train Loss: 0.6958, Test Loss: 0.6917
Epoch 5/500, Train Loss: 0.6918, Test Loss: 0.6913
Epoch 6/500, Train Loss: 0.6916, Test Loss: 0.6951
Epoch 7/500, Train Loss: 0.6963, Test Loss: 0.6911
Epoch 8/500, Train Loss: 0.6906, Test Loss: 0.6910
Epoch 9/500, Train Loss: 0.6907, Test Loss: 0.6919
Epoch 10/500, Train Loss: 0.6885, Test Loss: 0.6855
Epoch 11/500, Train Loss: 0.6847, Test Loss: 0.7362
Epoch 12/500, Train Loss: 0.6990, Test Loss: 0.7044
Epoch 13/500, Train Loss: 0.6960, Test Loss: 0.6905
Epoch 14/500, Train Loss: 0.6917, Test Loss: 0.6914
Epoch 15/500, Train Loss: 0.6907, Test Loss: 0.6909
Epoch 16/500, Train Loss: 0.6886, Test Loss: 0.6939
Epoch 17/500, Train Loss: 0.6925, Test Loss: 0.6889
Epoch 18/500, Train Loss: 0.6914, Test Loss: 0.6892
Epoch 19/500, Train Loss: 0.6882, Test Loss: 0.6883
Epoch 20/500, Train L

In [6]:
# 测试模型
# 加载模型
model.load_state_dict(torch.load('voltage_cnn_model.pth', weights_only=True))
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probability = torch.sigmoid(outputs)  # Convert logit to probability
        prediction = (probability > 0.5).float()
        total += labels.size(0)
        correct += (prediction == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probability = torch.sigmoid(outputs)  # Convert logit to probability
        prediction = (probability > 0.5).float()
        total += labels.size(0)
        correct += (prediction == labels).sum().item()

print(f"Train Accuracy: {100 * correct / total:.4f}%")

Test Accuracy: 100.00%
Train Accuracy: 100.0000%


In [7]:
# Calculate recall, F1 score, and AUC for the test set
test_labels = []
test_predictions = []
test_probabilities = []

with torch.no_grad():
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probability = torch.sigmoid(outputs)  # Convert logit to probability
        prediction = (probability > 0.5).float()
        
        test_labels.extend(labels.cpu().numpy())
        test_predictions.extend(prediction.cpu().numpy())
        test_probabilities.extend(probability.cpu().numpy())

test_labels = np.array(test_labels)
test_predictions = np.array(test_predictions)
test_probabilities = np.array(test_probabilities)

recall = recall_score(test_labels, test_predictions)
f1 = f1_score(test_labels, test_predictions)
auc = roc_auc_score(test_labels, test_probabilities)

print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:}")

Recall: 1.0000
F1 Score: 1.0000
AUC: 1.0
